In [1]:
import pandas as pd
import joblib
from langchain.tools import StructuredTool
from langchain_experimental.plan_and_execute import PlanAndExecute, load_chat_planner, load_agent_executor
from langchain.schema import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from sklearn.model_selection import train_test_split
import os
import json
import time

/Users/mahsaamani/Downloads/Saarlanduni/DataScience/UnveilingHospitalCostDrivers/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Load data
data = pd.read_csv('../data/Hospital_Inpatient_Discharges__SPARCS_De-Identified___2022_20250423.csv')
data.head()

/var/folders/s1/z7p70y8n35lcdk9lq0wvfxz00000gq/T/ipykernel_55785/1113152288.py:2: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('../data/Hospital_Inpatient_Discharges__SPARCS_De-Identified___2022_20250423.csv')


,Hospital Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code - 3 digits,Gender,Race,Ethnicity,...,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Payment Typology 2,Payment Typology 3,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs
0,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,50 to 69,107,F,White,Not Span/Hispanic,...,Major,Major,Medical,Medicaid,NaN,NaN,NaN,Y,"51,514.62","7,552.54"
1,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,18 to 29,104,M,Black/African American,Spanish/Hispanic,...,Moderate,Minor,Medical,Medicaid,NaN,NaN,NaN,Y,"25,370.86","3,469.55"
2,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,18 to 29,104,F,Other Race,Spanish/Hispanic,...,Minor,Minor,Medical,Medicaid,NaN,NaN,NaN,N,"23,876.78","6,180.33"
3,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,18 to 29,100,F,Black/African American,Not Span/Hispanic,...,Moderate,Minor,Medical,Medicaid,NaN,NaN,NaN,Y,"43,319.05","12,588.93"
4,New York City,Bronx,7000006.0,1168.0,Montefiore Medical Center-Wakefield Hospital,18 to 29,104,M,Other Race,Spanish/Hispanic,...,Moderate,Moderate,Medical,Medicaid,NaN,NaN,NaN,Y,"40,266.23","10,355.99"


In [3]:
import json

with open("../data/data_info.json", "r") as f:
    data_info = json.load(f)

In [4]:
for col in data.columns:
    if data[col].isna().any():
        print(col, data[col].isna().sum())

Hospital Service Area 5390
Hospital County 5390
Operating Certificate Number 5961
Permanent Facility Id 5390
Zip Code - 3 digits 41227
CCSR Procedure Code 582815
CCSR Procedure Description 582815
APR Severity of Illness Description 636
APR Risk of Mortality 636
Payment Typology 2 1121497
Payment Typology 3 1814362
Birth Weight 1894796


In [5]:
# Handle missing data and changing column type
for col_name, info in data_info.items():
    print(col_name)
    if info["type"] == "int64":
        if col_name == "Zip Code - 3 digits":
            data[col_name] = data[col_name].replace("OOS", 000)
            data[col_name] = data[col_name].fillna(000)
        if col_name == "Length of Stay":
            data[col_name] = data[col_name].replace("120 +", 121)
        if col_name == "Birth Weight":
            data[col_name] = data[col_name].replace("UNKN", -1)
            data[col_name] = data[col_name].fillna(-1)
        else:
            data[col_name] = data[col_name].fillna(-1)
            
        data[col_name] = pd.to_numeric(data[col_name]).astype('int')
        
    elif info["type"] == "str":
        data[col_name] = data[col_name].fillna("Unknown")
        data[col_name] = data[col_name].astype(str)

    elif info["type"] == "float64":
        data[col_name] = data[col_name].str.replace(',', '')
        data[col_name] = pd.to_numeric(data[col_name]).astype('float')


Hospital Service Area
Hospital County
Operating Certificate Number
Permanent Facility Id
Facility Name
Age Group
Zip Code - 3 digits
Gender
Race
Ethnicity
Length of Stay
Type of Admission
Patient Disposition
Discharge Year
CCSR Diagnosis Code
CCSR Diagnosis Description
CCSR Procedure Code
CCSR Procedure Description
APR DRG Code
APR DRG Description
APR MDC Code
APR MDC Description
APR Severity of Illness Code
APR Severity of Illness Description
APR Risk of Mortality
APR Medical Surgical Description
Payment Typology 1
Payment Typology 2
Payment Typology 3
Birth Weight
Emergency Department Indicator
Total Charges
Total Costs


In [6]:
for col in data.columns:
    if data[col].isna().any():
        print(col, data[col].isna().sum())

In [7]:
for col in data.columns:
    print(col, data[col].dtype)

Hospital Service Area object
Hospital County object
Operating Certificate Number int64
Permanent Facility Id int64
Facility Name object
Age Group object
Zip Code - 3 digits int64
Gender object
Race object
Ethnicity object
Length of Stay int64
Type of Admission object
Patient Disposition object
Discharge Year int64
CCSR Diagnosis Code object
CCSR Diagnosis Description object
CCSR Procedure Code object
CCSR Procedure Description object
APR DRG Code int64
APR DRG Description object
APR MDC Code int64
APR MDC Description object
APR Severity of Illness Code object
APR Severity of Illness Description object
APR Risk of Mortality object
APR Medical Surgical Description object
Payment Typology 1 object
Payment Typology 2 object
Payment Typology 3 object
Birth Weight int64
Emergency Department Indicator object
Total Charges float64
Total Costs float64


In [8]:
# Identify categorical columns and convert them to categories
categorical_columns = []
for col in data.columns:
    if data[col].dtype == "object":
        data[col] = data[col].astype('category')

In [9]:
X = data.drop(columns=['Total Costs', ])
y = data['Total Costs']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
model = joblib.load("../models/lgb_model.pkl")

In [12]:
# load shap explainer
explainer = joblib.load('../models/shap_explainer.pkl')

/Users/mahsaamani/Downloads/Saarlanduni/DataScience/UnveilingHospitalCostDrivers/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
SYSTEM_PROMPT = '''
You are an expert in healthcare cost optimization and hospital operations strategy.

Your role is to analyze structured data from individual inpatient hospital cases. Each feature includes:
- The data type (e.g., int, float, str)
- The observed value for the case
- The SHAP value, which quantifies how much that feature contributed to the total inpatient cost
- A list of possible values the feature can take (if available)

Your goal is to:
1. Identify which features are the most significant cost drivers.
2. Propose feasible, actionable strategies to reduce their cost impact, based on the available options.
3. Ensure your recommendations do not compromise—ideally improve—the quality of care.

Respond only with specific, pragmatic suggestions tailored to the data provided.
'''

USER_PROMPT = f'''
You are given structured data for a single inpatient case.

Each feature contains:
- "type": the data type
- "value": the observed value
- "shap value": contribution to the total inpatient cost
- "other options": the possible values for that feature (if any)

Your tasks:
1. For each feature, suggest a specific, feasible strategy to reduce its cost impact, using the "other options" where applicable to that patient. If there is no strategy, leave this empty.
2. Conclude with a short summary (2–3 sentences) explaining the overall cost-reduction logic you applied.
3. Return your response in this JSON format only (no additional text):

{{
  "Feature Name 1": "strategy",
  "Feature Name 2": "strategy",
  ...
  "Summary": "your overall summary"
}}

Data:
###patient_info###
'''

In [18]:
patient_dict = {
    "Hospital Service Area": "New York City",
    "Hospital County": "Kings",
    "Operating Certificate Number": 7001009,
    "Permanent Facility Id": 1294,
    "Facility Name": "Coney Island Hospital",
    "Age Group": "18 to 29",
    "Zip Code - 3 digits": 112,
    "Gender": "F",
    "Race": "Black/African American",
    "Ethnicity": "Not Span/Hispanic",
    "Length of Stay": 1,
    "Type of Admission": "Emergency",
    "Patient Disposition": "Home or Self Care",
    "Discharge Year": 2022,
    "CCSR Diagnosis Code": "INF012",
    "CCSR Diagnosis Description": "COVID-19",
    "CCSR Procedure Code": "ADM015",
    "CCSR Procedure Description": "ADMINISTRATION OF ANTIBIOTICS",
    "APR DRG Code": 137,
    "APR DRG Description": "MAJOR RESPIRATORY INFECTIONS AND INFLAMMATIONS",
    "APR MDC Code": 4,
    "APR MDC Description": "DISEASES AND DISORDERS OF THE RESPIRATORY SYSTEM",
    "APR Severity of Illness Code": 3,
    "APR Severity of Illness Description": "Major",
    "APR Risk of Mortality": "Moderate",
    "APR Medical Surgical Description": "Medical",
    "Payment Typology 1": "Medicaid",
    "Payment Typology 2": "Unknown",
    "Payment Typology 3": "Unknown",
    "Birth Weight": -1,
    "Emergency Department Indicator": "Y",
    "Total Charges": 8803.14
}

In [ ]:
shared_memory = {}
explainer = joblib.load('../models/shap_explainer.pkl')


def extract_shap_info():
    patient_df = pd.DataFrame(patient_dict, index=[0])
    for col in data.select_dtypes(['category']).columns:
        patient_df[col] = pd.Categorical(patient_df[col], categories=data[col].cat.categories)

    shared_memory["current cost"] = model.predict(patient_df).item()
    shap_values = explainer(patient_df)
    shap_info = {}
    for i, col in enumerate(patient_df.columns):
        shap_info[col] = {
            "type": data_info[col]["type"],
            "value": shap_values.data[0][i],
            "shap value": shap_values.values[0][i],
            "other options": data_info[col].get("options", [])
        }
    return json.dumps(shap_info)


def suggest_strategies(shap_info: str):
    shap_info = json.dumps(shap_info) if isinstance(shap_info, dict) else shap_info
    response = llm.invoke(
    [
        HumanMessage(
            content=USER_PROMPT.replace(
                "###patient_info###", json.dumps(shap_info)
            )
        ),
        SystemMessage(content=SYSTEM_PROMPT),
    ]
    )
    strategies = response.content.strip()
    return json.dumps(strategies)


def cost_predition(strategies: dict):
    strategies = json.loads(strategies) if isinstance(strategies, str) else strategies
    shared_memory["suggested strategies"] = strategies
    min_cost = shared_memory["current cost"]
    for col, method in strategies.items():
        if col == "Summary":
            continue
        info = data_info[col]
        if method != None and len(method) > 5:
            if info["type"] == "str" and "options" in info:
                # print(col, info["options"])
                for option in info["options"]:
                    updated_patient_dict = patient_dict.copy()
                    # apply changes
                    updated_patient_dict[col] = option
                    updated_patient_df = pd.DataFrame(updated_patient_dict, index=[0])
                    for col_n in data.select_dtypes(['category']).columns:
                        updated_patient_df[col_n] = pd.Categorical(updated_patient_df[col_n], categories=data[col_n].cat.categories)
                    cost = model.predict(updated_patient_df).item()
                    if cost <= min_cost:
                        min_cost = cost
                        shared_memory["new cost"] = min_cost
                        shared_memory["target feature"] = col
                        shared_memory["target strategy"] = option
    print(shared_memory)
    return json.dumps(shared_memory)

            
# --- LLM Setup ---
gemini_api_key = os.getenv("GEMINI_API_KEY")
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", api_key=gemini_api_key)

# --- Tool Collection and Agent Setup ---
tools = [
    StructuredTool.from_function(name="ExtractSHAPInfo", func=extract_shap_info, description="Compute SHAP values and return attributions"),
    StructuredTool.from_function(name="SuggestStrategies", func=suggest_strategies, description="Suggest realistic cost-reducing strategies for given features"),
    StructuredTool.from_function(name="CostPredition", func=cost_predition, description="Predict the cost after applying the strategies", return_direct=True),
]


planner = load_chat_planner(llm)
executor = load_agent_executor(llm=llm, tools=tools, verbose=False)
agent = PlanAndExecute(planner=planner, executor=executor, verbose=False, input_key="input")

# --- Execute Instruction ---
if __name__ == "__main__":
    prompt = f"""
        You are an expert in healthcare cost optimization and hospital operations strategy.

        Execute these steps using the available tools:
        
        1. Run ExtractSHAPInfo with the result.
        2. Run SuggestStrategies with the result.
        3. Run CostPredition with suggested strategies.
        Return the final recommended strategies.
        """
    
    max_retries = 10
    for attempt in range(max_retries):
        try:
            output = agent.invoke({"input": prompt})
            # json_output = json.loads(output) 
            print("Success!")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(10)
    else:
        print("All retries failed.")


{'current cost': 4451.418289105391, 'suggested strategies': {'Hospital Service Area': "Transfer the patient to a hospital located in a lower-cost service area such as 'Capital/Adirondack'.", 'Hospital County': 'Transfer the patient to a hospital in a lower-cost county such as Albany.', 'Facility Name': 'Transfer the patient to a lower-cost facility.', 'Length of Stay': 'Reduce the length of stay by 1 day.', 'Type of Admission': 'Avoid emergency admissions.', 'Patient Disposition': "Discharge patient to 'Home w/ Home Health Services'.", 'Emergency Department Indicator': 'Avoid emergency room visits.', 'Total Charges': 'Reduce overall charges by 10%.'}, 'new cost': -780.7799637453439, 'target feature': 'Facility Name', 'target strategy': 'Long Island Jewish Valley Stream'}
{'current cost': 4451.418289105391, 'suggested strategies': {'Hospital Service Area': "Transfer the patient to a hospital located in a lower-cost service area such as 'Capital/Adirondack'.", 'Hospital County': 'Transfe

In [20]:
shared_memory, output["output"]

({'current cost': 4451.418289105391,
  'suggested strategies': {'Hospital Service Area': "Transfer the patient to a hospital located in a lower-cost service area such as 'Capital/Adirondack'.",
   'Hospital County': 'Transfer the patient to a hospital in a lower-cost county such as Albany.',
   'Facility Name': 'Transfer the patient to a lower-cost facility.',
   'Length of Stay': 'Reduce the length of stay by 1 day.',
   'Type of Admission': 'Avoid emergency admissions.',
   'Patient Disposition': "Discharge patient to 'Home w/ Home Health Services'.",
   'Emergency Department Indicator': 'Avoid emergency room visits.',
   'Total Charges': 'Reduce overall charges by 10%'},
  'new cost': -780.7799637453439,
  'target feature': 'Facility Name',
  'target strategy': 'Long Island Jewish Valley Stream'},
 "Here are the recommended cost-reducing strategies:\n\n*   **Hospital Service Area:** Transfer the patient to a hospital located in a lower-cost service area such as 'Capital/Adirondack'.

# Evaluation

In [32]:
patient_dict = X_test.iloc[1].to_dict()
shared_memory = {}
explainer = joblib.load('../models/shap_explainer.pkl')


def extract_shap_info():
    patient_df = pd.DataFrame(patient_dict, index=[0])
    for col in data.select_dtypes(['category']).columns:
        patient_df[col] = pd.Categorical(patient_df[col], categories=data[col].cat.categories)

    shared_memory["current cost"] = model.predict(patient_df).item()
    shap_values = explainer(patient_df)
    shap_info = {}
    for i, col in enumerate(patient_df.columns):
        shap_info[col] = {
            "type": data_info[col]["type"],
            "value": shap_values.data[0][i],
            "shap value": shap_values.values[0][i],
            "other options": data_info[col].get("options", [])
        }
    return json.dumps(shap_info)


def suggest_strategies(shap_info: str):
    shap_info = json.dumps(shap_info) if isinstance(shap_info, dict) else shap_info
    response = llm.invoke(
    [
        HumanMessage(
            content=USER_PROMPT.replace(
                "###patient_info###", json.dumps(shap_info)
            )
        ),
        SystemMessage(content=SYSTEM_PROMPT),
    ]
    )
    strategies = response.content.strip()
    return json.dumps(strategies)


def cost_predition(strategies: dict):
    strategies = json.loads(strategies) if isinstance(strategies, str) else strategies
    shared_memory["suggested strategies"] = strategies
    min_cost = shared_memory["current cost"]
    for col, method in strategies.items():
        if col == "Summary":
            continue
        info = data_info[col]
        if method != None and len(method) > 5:
            if info["type"] == "str" and "options" in info:
                # print(col, info["options"])
                for option in info["options"]:
                    updated_patient_dict = patient_dict.copy()
                    # apply changes
                    updated_patient_dict[col] = option
                    updated_patient_df = pd.DataFrame(updated_patient_dict, index=[0])
                    for col_n in data.select_dtypes(['category']).columns:
                        updated_patient_df[col_n] = pd.Categorical(updated_patient_df[col_n], categories=data[col_n].cat.categories)
                    cost = model.predict(updated_patient_df).item()
                    if cost <= min_cost:
                        min_cost = cost
                        shared_memory["new cost"] = min_cost
                        shared_memory["target feature"] = col
                        shared_memory["target strategy"] = option
    print(shared_memory)
    return json.dumps(shared_memory)

            
# --- LLM Setup ---
gemini_api_key = os.getenv("GEMINI_API_KEY")
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", api_key=gemini_api_key)

# --- Tool Collection and Agent Setup ---
tools = [
    StructuredTool.from_function(name="ExtractSHAPInfo", func=extract_shap_info, description="Compute SHAP values and return attributions"),
    StructuredTool.from_function(name="SuggestStrategies", func=suggest_strategies, description="Suggest realistic cost-reducing strategies for given features"),
    StructuredTool.from_function(name="CostPredition", func=cost_predition, description="Predict the cost after applying the strategies", return_direct=True),
]


planner = load_chat_planner(llm)
executor = load_agent_executor(llm=llm, tools=tools, verbose=False)
agent = PlanAndExecute(planner=planner, executor=executor, verbose=False, input_key="input")

# --- Execute Instruction ---
if __name__ == "__main__":
    prompt = f"""
        You are an expert in healthcare cost optimization and hospital operations strategy.

        Execute these steps using the available tools:
        
        1. Run ExtractSHAPInfo with the result.
        2. Run SuggestStrategies with the result.
        3. Run CostPredition with suggested strategies.
        Return the final recommended strategies.
        """
    
    max_retries = 10
    for attempt in range(max_retries):
        try:
            output = agent.invoke({"input": prompt})
            # json_output = json.loads(output) 
            print("Success!")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(10)
    else:
        print("All retries failed.")


{'current cost': 20220.32719287887, 'suggested strategies': {'Hospital Service Area': 'If feasible and without compromising care, consider transferring the patient to a hospital within a lower-cost service area like Capital/Adirondack.', 'Hospital County': 'If appropriate and without compromising care, transfer the patient to a hospital in a lower-cost county.', 'Facility Name': "Transfer the patient to a lower-cost facility, if medically appropriate and feasible.  Consider facilities with negative SHAP values or those listed earlier in the 'other options'.", 'Length of Stay': 'Optimize care pathways to reduce the length of stay, while ensuring appropriate and safe discharge.', 'Type of Admission': 'Explore alternative admission pathways (e.g., urgent care) if the condition allows, to potentially avoid the higher costs associated with emergency admissions.', 'Patient Disposition': "Discharge to 'Home w/ Home Health Services' instead of 'Short-term Hospital' if appropriate to avoid read

In [33]:
shared_memory, output["output"]

({'current cost': 20220.32719287887,
  'suggested strategies': {'Hospital Service Area': 'If feasible and without compromising care, consider transferring the patient to a hospital within a lower-cost service area like Capital/Adirondack.',
   'Hospital County': 'If appropriate and without compromising care, transfer the patient to a hospital in a lower-cost county.',
   'Facility Name': "Transfer the patient to a lower-cost facility, if medically appropriate and feasible.  Consider facilities with negative SHAP values or those listed earlier in the 'other options'.",
   'Length of Stay': 'Optimize care pathways to reduce the length of stay, while ensuring appropriate and safe discharge.',
   'Type of Admission': 'Explore alternative admission pathways (e.g., urgent care) if the condition allows, to potentially avoid the higher costs associated with emergency admissions.',
   'Patient Disposition': "Discharge to 'Home w/ Home Health Services' instead of 'Short-term Hospital' if appropr

In [34]:
for pindex in range(10):
    patient_dict = X_test.iloc[pindex].to_dict()
    shared_memory = {}
    explainer = joblib.load('../models/shap_explainer.pkl')


    def extract_shap_info():
        patient_df = pd.DataFrame(patient_dict, index=[0])
        for col in data.select_dtypes(['category']).columns:
            patient_df[col] = pd.Categorical(patient_df[col], categories=data[col].cat.categories)

        shared_memory["current cost"] = model.predict(patient_df).item()
        shap_values = explainer(patient_df)
        shap_info = {}
        for i, col in enumerate(patient_df.columns):
            shap_info[col] = {
                "type": data_info[col]["type"],
                "value": shap_values.data[0][i],
                "shap value": shap_values.values[0][i],
                "other options": data_info[col].get("options", [])
            }
        return json.dumps(shap_info)


    def suggest_strategies(shap_info: str):
        shap_info = json.dumps(shap_info) if isinstance(shap_info, dict) else shap_info
        response = llm.invoke(
        [
            HumanMessage(
                content=USER_PROMPT.replace(
                    "###patient_info###", json.dumps(shap_info)
                )
            ),
            SystemMessage(content=SYSTEM_PROMPT),
        ]
        )
        strategies = response.content.strip()
        return json.dumps(strategies)


    def cost_predition(strategies: dict):
        strategies = json.loads(strategies) if isinstance(strategies, str) else strategies
        shared_memory["suggested strategies"] = strategies
        min_cost = shared_memory["current cost"]
        for col, method in strategies.items():
            if col == "Summary":
                continue
            info = data_info[col]
            if method != None and len(method) > 5:
                if info["type"] == "str" and "options" in info:
                    # print(col, info["options"])
                    for option in info["options"]:
                        updated_patient_dict = patient_dict.copy()
                        # apply changes
                        updated_patient_dict[col] = option
                        updated_patient_df = pd.DataFrame(updated_patient_dict, index=[0])
                        for col_n in data.select_dtypes(['category']).columns:
                            updated_patient_df[col_n] = pd.Categorical(updated_patient_df[col_n], categories=data[col_n].cat.categories)
                        cost = model.predict(updated_patient_df).item()
                        if cost < min_cost:
                            min_cost = cost
                            shared_memory["new cost"] = min_cost
                            shared_memory["target feature"] = col
                            shared_memory["target strategy"] = option
        # print(shared_memory)
        return json.dumps(shared_memory)

                
    # --- LLM Setup ---
    gemini_api_key = os.getenv("GEMINI_API_KEY")
    llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", api_key=gemini_api_key)

    # --- Tool Collection and Agent Setup ---
    tools = [
        StructuredTool.from_function(name="ExtractSHAPInfo", func=extract_shap_info, description="Compute SHAP values and return attributions"),
        StructuredTool.from_function(name="SuggestStrategies", func=suggest_strategies, description="Suggest realistic cost-reducing strategies for given features"),
        StructuredTool.from_function(name="CostPredition", func=cost_predition, description="Predict the cost after applying the strategies", return_direct=True),
    ]


    planner = load_chat_planner(llm)
    executor = load_agent_executor(llm=llm, tools=tools, verbose=False)
    agent = PlanAndExecute(planner=planner, executor=executor, verbose=False, input_key="input")

    # --- Execute Instruction ---
    if __name__ == "__main__":
        prompt = f"""
            You are an expert in healthcare cost optimization and hospital operations strategy.

            Execute these steps using the available tools:
            
            1. Run ExtractSHAPInfo with the result.
            2. Run SuggestStrategies with the result.
            3. Run CostPredition with suggested strategies.
            Return the final recommended strategies.
            """
        
        max_retries = 10
        for attempt in range(max_retries):
            try:
                output = agent.invoke({"input": prompt})
                # json_output = json.loads(output) 
                print("Success!")
                break
            except Exception as e:
                print(f"Attempt {attempt + 1} failed.")
                time.sleep(10)
        else:
            print("All retries failed.")
    print(f"________________{pindex}______________")
    print(shared_memory, output["output"])


Attempt 1 failed.
Attempt 2 failed.
Attempt 3 failed.
Success!
________________0______________
{'current cost': 4241.292038287831, 'suggested strategies': {'Length of Stay': 'Reduce length of stay by 1 day', 'CCSR Procedure Code': 'Documented procedure code', 'APR Severity of Illness Code': "Coded as '0'", 'APR Risk of Mortality': "Coded as 'Minor'", 'Emergency Department Indicator': "Changed to 'N'", 'Total Charges': 'Reduced by 10%'}, 'new cost': 4241.292038287831, 'target feature': 'Emergency Department Indicator', 'target strategy': 'Y'} {"current cost": 4241.292038287831, "suggested strategies": {"Length of Stay": "Reduce length of stay by 1 day", "CCSR Procedure Code": "Documented procedure code", "APR Severity of Illness Code": "Coded as '0'", "APR Risk of Mortality": "Coded as 'Minor'", "Emergency Department Indicator": "Changed to 'N'", "Total Charges": "Reduced by 10%"}, "new cost": 4241.292038287831, "target feature": "Emergency Department Indicator", "target strategy": "Y"}

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 48
}
].


Attempt 1 failed.
Attempt 2 failed.
Success!
________________2______________
{'current cost': 1936.4713537395544, 'suggested strategies': {'Hospital Service Area': 'Shift care to a hospital within the Capital/Adirondack service area if feasible.', 'Hospital County': 'Shift care to a hospital within Albany County if feasible.', 'Facility Name': 'Shift care to Montefiore Med Center - Jack D Weiler Hosp of A Einstein College Div if feasible.', 'Length of Stay': 'Reduce length of stay if medically appropriate.', 'APR Severity of Illness Code': "If possible and medically appropriate, ensure the severity of illness is accurately coded and not overstated. If truly minor, confirm the code is '0'.", 'Emergency Department Indicator': 'Ensure emergency department use was necessary and appropriate.', 'Total Charges': 'Review and negotiate total charges to ensure accuracy and fair pricing.'}, 'new cost': 1160.2724618413404, 'target feature': 'Facility Name', 'target strategy': 'New York-Presbyteria

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 57
}
].


Attempt 1 failed.
Attempt 2 failed.
Attempt 3 failed.
Attempt 4 failed.
Attempt 5 failed.
Attempt 6 failed.
Success!
________________9______________
{'current cost': 6864.598790790437, 'suggested strategies': {'Hospital Service Area': 'Transfer the patient, if medically appropriate and feasible, to a hospital in a lower-cost service area (e.g., Capital/Adirondack, Central NY).', 'Hospital County': 'Transfer the patient, if medically appropriate and feasible, to a hospital in a lower-cost county.', 'Facility Name': 'Transfer the patient, if medically appropriate and feasible, to a lower-cost facility within the same county or service area.', 'Age Group': 'No strategy.', 'Gender': 'No strategy.', 'Race': 'No strategy.', 'Ethnicity': 'No strategy.', 'Type of Admission': 'Ensure appropriate utilization of resources during the emergency admission to minimize length of stay and complications.', 'Patient Disposition': "If clinically appropriate, explore options for discharge to 'Home w/ Home 

In [37]:
yes = 0
for pindex in range(50):
    patient_dict = X_test.iloc[pindex].to_dict()
    shared_memory = {}
    explainer = joblib.load('../models/shap_explainer.pkl')


    def extract_shap_info():
        patient_df = pd.DataFrame(patient_dict, index=[0])
        for col in data.select_dtypes(['category']).columns:
            patient_df[col] = pd.Categorical(patient_df[col], categories=data[col].cat.categories)

        shared_memory["current cost"] = model.predict(patient_df).item()
        shap_values = explainer(patient_df)
        shap_info = {}
        for i, col in enumerate(patient_df.columns):
            shap_info[col] = {
                "type": data_info[col]["type"],
                "value": shap_values.data[0][i],
                "shap value": shap_values.values[0][i],
                "other options": data_info[col].get("options", [])
            }
        return json.dumps(shap_info)


    def suggest_strategies(shap_info: str):
        shap_info = json.dumps(shap_info) if isinstance(shap_info, dict) else shap_info
        response = llm.invoke(
        [
            HumanMessage(
                content=USER_PROMPT.replace(
                    "###patient_info###", json.dumps(shap_info)
                )
            ),
            SystemMessage(content=SYSTEM_PROMPT),
        ]
        )
        strategies = response.content.strip()
        return json.dumps(strategies)


    def cost_predition(strategies: dict):
        strategies = json.loads(strategies) if isinstance(strategies, str) else strategies
        shared_memory["suggested strategies"] = strategies
        min_cost = shared_memory["current cost"]
        for col, method in strategies.items():
            if col == "Summary":
                continue
            info = data_info[col]
            if method != None and len(method) > 5:
                if info["type"] == "str" and "options" in info:
                    # print(col, info["options"])
                    for option in info["options"]:
                        updated_patient_dict = patient_dict.copy()
                        # apply changes
                        updated_patient_dict[col] = option
                        updated_patient_df = pd.DataFrame(updated_patient_dict, index=[0])
                        for col_n in data.select_dtypes(['category']).columns:
                            updated_patient_df[col_n] = pd.Categorical(updated_patient_df[col_n], categories=data[col_n].cat.categories)
                        cost = model.predict(updated_patient_df).item()
                        if cost < min_cost:
                            min_cost = cost
                            shared_memory["new cost"] = min_cost
                            shared_memory["target feature"] = col
                            shared_memory["target strategy"] = option
        # print(shared_memory)
        return json.dumps(shared_memory)

                
    # --- LLM Setup ---
    gemini_api_key = os.getenv("GEMINI_API_KEY")
    llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", api_key=gemini_api_key)

    # --- Tool Collection and Agent Setup ---
    tools = [
        StructuredTool.from_function(name="ExtractSHAPInfo", func=extract_shap_info, description="Compute SHAP values and return attributions"),
        StructuredTool.from_function(name="SuggestStrategies", func=suggest_strategies, description="Suggest realistic cost-reducing strategies for given features"),
        StructuredTool.from_function(name="CostPredition", func=cost_predition, description="Predict the cost after applying the strategies", return_direct=True),
    ]


    planner = load_chat_planner(llm)
    executor = load_agent_executor(llm=llm, tools=tools, verbose=False)
    agent = PlanAndExecute(planner=planner, executor=executor, verbose=False, input_key="input")

    # --- Execute Instruction ---
    if __name__ == "__main__":
        prompt = f"""
            You are an expert in healthcare cost optimization and hospital operations strategy.

            Execute these steps using the available tools:
            
            1. Run ExtractSHAPInfo with the result.
            2. Run SuggestStrategies with the result.
            3. Run CostPredition with suggested strategies.
            Return the final recommended strategies.
            """
        
        max_retries = 10
        for attempt in range(max_retries):
            try:
                output = agent.invoke({"input": prompt})
                # json_output = json.loads(output) 
                print("Success!")
                break
            except Exception as e:
                print(f"Attempt {attempt + 1} failed.")
                time.sleep(10)
        else:
            print("All retries failed.")

    if "new cost" in shared_memory:
        yes += 1
    print(f"________________{pindex}______________")
    print(shared_memory, output["output"])



Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 1000
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 27
}
].


Attempt 1 failed.


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 1000
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 15
}
].


Attempt 2 failed.


KeyboardInterrupt: 

In [36]:
yes

21